# 实践项目 09：ECG R 峰检测与跨记录验证

本项目使用 PhysioNet MIT-BIH Arrhythmia Database 中 7 条独立记录的前 5 分钟。记录 100 只用于选择检测参数；记录 101、103、105、106、108、109 保持参数冻结，用于检查算法在不同波形、节律和噪声条件下的表现。

## 任务总览

1. 核对记录、导联、采样率和人工心搏标注。
2. 使用 5—18 Hz 带通滤波和短时能量构造 R 峰候选信号。
3. 只在 record 100 比较阈值分位数和最小峰间距离。
4. 固定参数后逐条评价其余记录，报告 sensitivity、positive predictive value 和 F1。
5. 查看漏检较多的记录，说明固定规则的适用边界。

这里的评价对象是人工标注心搏附近是否检测到峰，不是心律失常分类或临床诊断。

In [ ]:
from pathlib import Path  # 管理数据和输出路径
import json  # 保存结果摘要
import numpy as np  # 处理 ECG 数组
import pandas as pd  # 整理逐记录指标
import matplotlib.pyplot as plt  # 绘制波形与指标图
from scipy.signal import butter, sosfiltfilt, find_peaks  # 滤波和峰值检测


def find_project_root():  # 从任意 Notebook 工作目录定位仓库根目录
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / 'experience' / 'data' / 'project-10').exists():
            return root
    return Path.cwd()


project_root = find_project_root()


def find_file(name):  # 兼容 Kaggle 数据集和本地仓库
    roots = [Path('/kaggle/input'), project_root, project_root / 'experience' / 'data' / 'project-10']
    for root in roots:
        if root.exists():
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'没有找到 {name}，请添加项目 09 数据集。')


bundle = np.load(find_file('mitdb_multirecord_5min.npz'), allow_pickle=True)  # 读取7条真实记录
record_ids = bundle['record_id'].astype(str)
signals = bundle['signal'].astype(np.float32)
sig_names = bundle['sig_name'].astype(str)
sampling_rates = bundle['fs'].astype(float)
annotations = bundle['ann_sample']
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else project_root / 'experience' / 'assets' / 'results' / 'project-10'
output_dir.mkdir(parents=True, exist_ok=True)
print({'records': record_ids.tolist(), 'signal_shape': signals.shape, 'sampling_rates': sampling_rates.tolist()})

## 任务 1：记录、导联与标注核对

逐条列出输入规模和人工标注数，明确 record 100 与冻结测试记录的用途。

In [ ]:
record_table = []  # TODO 1：逐条记录填写导联、样本数、时长和标注心搏数
lead_indexes = []
for index, record_id in enumerate(record_ids):
    names = sig_names[index].astype(str)
    # TODO：优先选择 MLII；没有 MLII 时使用第一个导联
    lead_index = None
    if lead_index is None:
        raise NotImplementedError('请完成导联选择')
    lead_indexes.append(lead_index)
    record_table.append({
        'record': record_id,
        'lead': names[lead_index],
        'samples': int(signals.shape[1]),
        'seconds': float(signals.shape[1] / sampling_rates[index]),
        'annotated_beats': int(len(annotations[index])),
    })
record_table = pd.DataFrame(record_table)
record_table

## 任务 2：滤波、短时能量与一对一匹配

带通滤波用于突出 QRS 相关频率，平方与移动平均把双向波形转为正的短时能量。一对一匹配避免多个检测峰重复匹配同一人工标注。

In [ ]:
def detection_energy(signal, fs):  # 构造用于峰值检测的短时能量
    sos = butter(3, [5, 18], btype='bandpass', fs=fs, output='sos')
    filtered = sosfiltfilt(sos, signal)
    squared = filtered ** 2
    width = max(1, int(0.08 * fs))
    energy = np.convolve(squared, np.ones(width) / width, mode='same')
    return filtered, energy


def one_to_one_match(detected, annotated, tolerance):  # 一对一匹配检测峰和人工标注
    detected = np.asarray(detected, dtype=int)
    annotated = np.asarray(annotated, dtype=int)
    i = j = true_positive = 0
    while i < len(detected) and j < len(annotated):
        delta = int(detected[i] - annotated[j])
        if abs(delta) <= tolerance:
            true_positive += 1
            i += 1
            j += 1
        elif delta < -tolerance:
            i += 1
        else:
            j += 1
    false_positive = len(detected) - true_positive
    false_negative = len(annotated) - true_positive
    return true_positive, false_positive, false_negative


def detect_record(index, quantile, min_distance_seconds):  # 使用同一规则处理任意记录
    fs = float(sampling_rates[index])
    lead = int(lead_indexes[index])
    filtered, energy = detection_energy(signals[index, :, lead].astype(float), fs)
    threshold = float(np.quantile(energy, quantile))
    peaks, _ = find_peaks(energy, height=threshold, distance=int(min_distance_seconds * fs))
    return filtered, energy, peaks

## 任务 3：只在 record 100 选择参数

参数搜索不能查看冻结测试记录的标签表现。选定参数后不再针对单条测试记录调整。

In [ ]:
development_index = int(np.where(record_ids == '100')[0][0])  # record 100 只用于参数选择
search_results = []
# TODO 2：比较 quantile=0.88—0.99 和 min_distance=0.20—0.30 秒
# 每组参数调用 detect_record 和 one_to_one_match，并保存 F1、FP、FN。
raise NotImplementedError('请完成 record 100 参数搜索')

## 任务 4：冻结参数后的跨记录评价

逐条报告指标，避免用总体高分掩盖某些记录上的漏检。

In [ ]:
rows = []
# TODO 3：使用 best 中冻结的参数逐条评价 7 条记录。
# record 100 标为 development，其余记录标为 frozen test。
# 汇总测试记录的 TP、FP、FN、sensitivity、PPV 和 F1。
raise NotImplementedError('请完成跨记录评价')

## 任务 5：错误记录与结论边界

查看最低 F1 记录并保存逐记录指标。固定规则的高分只支持本课程数据范围内的 R 峰检测，不支持疾病分类或临床应用结论。

In [ ]:
# TODO 4：绘制6条冻结测试记录的F1柱状图，并选择最低F1记录显示10秒波形、检测峰和人工标注。
# 保存 future09_multirecord_summary.png 和 future09_multirecord_result.json。
raise NotImplementedError('请完成结果图和结果摘要')